# HW2: финальный инференс на одной NVIDIA T4

Финальная конфигурация: beam=4, max_new_tokens=128, batch_size=32, length_penalty=0.6, FP16 autocast и static KV-cache.

Подключите inference-артефакт с model.pt, tokenizer.json, manifest.json, test/ и code_bundle.zip. Веса обучены с нуля на исходном train. Дополнительные данные и таргеты не используются.

Подготовительные ячейки копируют артефакт на локальный диск и устанавливают зависимости. Отдельный замер ниже включает запуск Python, загрузку модели, токенизацию всех 1919 вопросов, генерацию и запись JSONL. Процессу видна только одна T4.


In [ ]:
from pathlib import Path
import hashlib
import json
import os
import subprocess
import shutil
import sys
import time
import zipfile

# При необходимости задайте путь вручную, например после распаковки ZIP.
ARTIFACT_OVERRIDE = None
if ARTIFACT_OVERRIDE is None:
    candidates = [p.parent for p in Path('/kaggle/input').rglob('model.pt')
                  if (p.parent / 'tokenizer.json').exists() and (p.parent / 'test').is_dir()]
    assert len(candidates) == 1, f'Подключите один inference-артефакт; найдено {len(candidates)}'
    SOURCE_ARTIFACT = candidates[0]
else:
    SOURCE_ARTIFACT = Path(ARTIFACT_OVERRIDE)

ARTIFACT = Path('/kaggle/working/hw2_artifact')
ARTIFACT.mkdir(exist_ok=True)
for name in ['model.pt', 'tokenizer.json', 'manifest.json']:
    shutil.copyfile(SOURCE_ARTIFACT / name, ARTIFACT / name)
shutil.copytree(SOURCE_ARTIFACT / 'test', ARTIFACT / 'test', dirs_exist_ok=True)
PROJECT = Path('/kaggle/working/hw2_inference')
PROJECT.mkdir(exist_ok=True)
archive_path = SOURCE_ARTIFACT / 'code_bundle.zip'
if archive_path.exists():
    with zipfile.ZipFile(archive_path) as archive:
        for member in archive.infolist():
            assert (PROJECT / member.filename).resolve().is_relative_to(PROJECT.resolve())
        archive.extractall(PROJECT)
else:
    assert (SOURCE_ARTIFACT / 'inference.py').exists(), 'Добавьте code_bundle.zip в экспорт'
    bundle = json.loads((SOURCE_ARTIFACT / 'bundle_manifest.json').read_text())
    for relative in [*bundle['sha256'], 'bundle_manifest.json']:
        target = PROJECT / relative
        assert target.resolve().is_relative_to(PROJECT.resolve())
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(SOURCE_ARTIFACT / relative, target)
for relative, digest in json.loads((PROJECT / 'bundle_manifest.json').read_text())['sha256'].items():
    assert hashlib.sha256((PROJECT / relative).read_bytes()).hexdigest() == digest, relative
print('Артефакт на локальном диске:', ARTIFACT)


Устанавливаем зависимости из requirements-kaggle.txt. CUDA-версия PyTorch остаётся из окружения Kaggle. Эта подготовительная ячейка не входит в замер отдельного процесса генерации ниже.


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-input',
                '-r', str(PROJECT / 'requirements-kaggle.txt')], check=True)


Запускаем финальный beam=4 в новом Python-процессе. Установка зависимостей, копирование файлов и распаковка кода выполнены выше и не входят в этот замер. Время всего дочернего процесса включает импорты, загрузку весов, инициализацию CUDA, токенизацию исходных вопросов, генерацию, декодирование, запись файла и завершение процесса.


In [ ]:
OUTPUT = Path('/kaggle/working/predictions.jsonl')
REPORT = Path('/kaggle/working/inference_report.json')
env = dict(os.environ, CUDA_VISIBLE_DEVICES='0', TOKENIZERS_PARALLELISM='false', OMP_NUM_THREADS='1')
started = time.perf_counter()
subprocess.run([
    sys.executable, str(PROJECT / 'inference.py'),
    f'checkpoint={ARTIFACT / "model.pt"}', f'prepared_dir={ARTIFACT}',
    f'output_path={OUTPUT}', f'report_path={REPORT}',
    'device=cuda', 'require_t4=true', 'batch_size=32', 'max_new_tokens=128',
    'num_beams=4', 'length_penalty=0.6', 'amp=true', 'cache_type=static',
    'sort_by_length=true', 'compile_step=false', 'split=test', 'limit=null',
], cwd=PROJECT, env=env, check=True)
process_seconds = time.perf_counter() - started
report = json.loads(REPORT.read_text())
report['external_process_seconds'] = process_seconds
report['passes_required_benchmark'] = bool(report['passes_required_benchmark'] and process_seconds <= 180)
REPORT.write_text(json.dumps(report, indent=2, ensure_ascii=False))
print('Время внешнего процесса:', process_seconds)
print(json.dumps(report, indent=2, ensure_ascii=False))


Проверяем формат и полноту, не исполняя сгенерированные программы. `syntax_parse_rate` в отчёте не равен pass@1; pass@1 измеряет бот. Файл для отправки — `predictions.jsonl`.


In [ ]:
rows = [json.loads(line) for line in OUTPUT.read_text().splitlines()]
assert len(rows) == report['expected_split_examples']
assert all(set(row) == {'code'} and isinstance(row['code'], str) for row in rows)
assert report['passes_required_benchmark'], 'Лимит или условие одной T4 не выполнены'
assert process_seconds <= 180, 'Внешний процесс превысил 180 секунд'
print('Готово:', OUTPUT, 'задач:', len(rows))
